# 03. Combined Frequency-Severity Model Evaluation

This notebook performs the orchestration and evaluation of the combined Frequency-Severity pricing model. 

### Formula:
$$\text{expected\_annual\_cost} = \text{predicted\_frequency\_rate} \times \text{predicted\_severity}$$

where:
- $\text{predicted\_frequency\_rate}$: Expected claim count per year (annualized).
- $\text{predicted\_severity}$: Expected cost per claim.

In [ ]:
# If you get "ModuleNotFoundError: No module named 'statsmodels'" or other packages,
# please run this cell to install all required packages in your active Jupyter kernel:
# !pip install statsmodels xgboost scikit-learn joblib pandas numpy matplotlib seaborn

In [ ]:
# 1. Setup path and imports
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import load
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

# Add project root and ai-model-service to path
# Since the notebook is in ai-model-service/notebooks, the project root is two levels up ("../..")
project_root = Path("../..").resolve()
if str(project_root / "ai-model-service") not in sys.path:
    sys.path.append(str(project_root / "ai-model-service"))

import app.custom_models  # Register custom pipelines for joblib

%matplotlib inline
sns.set_theme(style="whitegrid")

## 2. Load Models and Metadata

In [ ]:
freq_model_path = project_root / "ml/artifacts/frequency_model.joblib"
sev_model_path = project_root / "ml/artifacts/severity_model.joblib"
freq_meta_path = project_root / "ml/artifacts/frequency_metadata.json"
sev_meta_path = project_root / "ml/artifacts/severity_metadata.json"

print(f"Loading models from {project_root}...")
freq_model = load(freq_model_path)
sev_model = load(sev_model_path)

print("Loading metadata...")
with open(freq_meta_path, "r") as f:
    freq_meta = json.load(f)
with open(sev_meta_path, "r") as f:
    sev_meta = json.load(f)

freq_features = freq_meta["featureList"]
sev_features = sev_meta["featureList"]

print(f"Frequency model: {freq_meta['modelVersion']}")
print(f"Severity model: {sev_meta['modelVersion']}")

## 3. Load and Split Master Dataset

In [ ]:
dataset_path = project_root / "data/generated/synthetic_insurance_claims.csv"
print(f"Loading dataset from {dataset_path}...")
df_raw = pd.read_csv(dataset_path)

# Filter to exposure >= 0.1 to avoid extreme division / tiny policies
df = df_raw[df_raw["exposure_time"] >= 0.1].copy()

# Split test set with same random state 42
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
print(f"Master set size: {len(df)}")
print(f"Holdout evaluation set size: {len(df_test)}")

## 4. Predict Frequency & Severity and Calculate Combined Premium

In [ ]:
print("Predicting frequency...")
pred_freq_count = freq_model.predict(df_test[freq_features])
predicted_frequency_rate = pred_freq_count / df_test["exposure_time"].values

print("Predicting severity...")
predicted_severity = sev_model.predict(df_test[sev_features])

print("Calculating combined predicted annual cost...")
predicted_annual_cost = predicted_frequency_rate * predicted_severity
actual_annual_cost = df_test["annual_claim_cost"].values / df_test["exposure_time"].values

# Clip predictions to avoid negative values
pred_clean = np.maximum(predicted_annual_cost, 0.0)

## 5. Metric Functions

In [ ]:
def normalized_gini(y_true, y_pred):
    actual = np.asarray(y_true, dtype=float)
    pred = np.asarray(y_pred, dtype=float)
    if len(actual) == 0 or np.all(actual == actual[0]):
        return 0.0

    def gini(values, scores):
        order = np.lexsort((np.arange(len(scores)), -scores))
        sorted_values = values[order]
        cumulative = np.cumsum(sorted_values)
        if cumulative[-1] == 0:
            return 0.0
        lorenz = cumulative / cumulative[-1]
        random = (np.arange(len(values)) + 1) / len(values)
        return float(np.sum(lorenz - random) / len(values))

    perfect = gini(actual, actual)
    return 0.0 if perfect == 0 else float(gini(actual, pred) / perfect)

def top_10_percent_lift(y_true, y_pred, exposure):
    actual = np.asarray(y_true, dtype=float)
    pred = np.asarray(y_pred, dtype=float)
    exp = np.asarray(exposure, dtype=float)
    
    if len(actual) == 0:
        return 0.0
        
    order = np.argsort(-pred)
    sorted_actual = actual[order]
    sorted_exp = exp[order]
    
    n_10 = max(1, int(len(actual) * 0.1))
    
    top_actual_cost = np.sum(sorted_actual[:n_10] * sorted_exp[:n_10])
    top_exposure = np.sum(sorted_exp[:n_10])
    top_rate = top_actual_cost / np.maximum(top_exposure, 1e-6)
    
    portfolio_actual_cost = np.sum(actual * exp)
    portfolio_exposure = np.sum(exp)
    portfolio_rate = portfolio_actual_cost / np.maximum(portfolio_exposure, 1e-6)
    
    if portfolio_rate == 0:
        return 0.0
        
    return float(top_rate / portfolio_rate)

## 6. Calculate Results

In [ ]:
mae = mean_absolute_error(actual_annual_cost, pred_clean)
gini = normalized_gini(actual_annual_cost, pred_clean)
lift = top_10_percent_lift(actual_annual_cost, pred_clean, df_test["exposure_time"].values)

actual_total_cost = df_test["annual_claim_cost"].sum()
pred_total_cost = np.sum(pred_clean * df_test["exposure_time"].values)
cal_ratio_weighted = actual_total_cost / pred_total_cost if pred_total_cost > 0 else 0.0

print(f"MAE: {mae:.4f}")
print(f"Normalized Gini: {gini:.4f}")
print(f"Top-10% Lift: {lift:.4f}")
print(f"Calibration Ratio (Weighted): {cal_ratio_weighted:.4f}")

## 7. Generate Visualizations

In [ ]:
# Plot 1: Actual vs Predicted Log1p
plt.figure(figsize=(8, 6))
sns.scatterplot(
    x=np.log1p(pred_clean),
    y=np.log1p(actual_annual_cost),
    alpha=0.3,
    color="#2b5c8f",
    edgecolor=None
)
max_val = max(np.log1p(pred_clean).max(), np.log1p(actual_annual_cost).max())
plt.plot([0, max_val], [0, max_val], color="#e74c3c", linestyle="--", linewidth=2, label="Perfect Calibration (y = x)")
plt.xlabel("Log1p(Predicted Annual Cost)")
plt.ylabel("Log1p(Actual Annual Cost)")
plt.title("Actual vs. Predicted Annual Cost (Log1p)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Deciles of Risk
df_eval = df_test.copy()
df_eval["pred_annual_cost"] = pred_clean
df_eval["actual_annual_cost"] = actual_annual_cost
df_eval["decile"] = pd.qcut(df_eval["pred_annual_cost"], 10, labels=False, duplicates='drop') + 1

decile_summary = df_eval.groupby("decile").apply(lambda g: pd.Series({
    "actual": np.sum(g["annual_claim_cost"]) / np.sum(g["exposure_time"]),
    "predicted": np.sum(g["pred_annual_cost"] * g["exposure_time"]) / np.sum(g["exposure_time"])
}), include_groups=False).reset_index()

plt.figure(figsize=(10, 6))
x_indices = np.arange(len(decile_summary))
width = 0.35
plt.bar(x_indices - width/2, decile_summary["actual"], width, label="Actual Annual Cost", color="#34495e")
plt.bar(x_indices + width/2, decile_summary["predicted"], width, label="Predicted Annual Cost", color="#2ecc71")
plt.xlabel("Risk Decile (1 = Lowest Risk, 10 = Highest Risk)")
plt.ylabel("Annual Cost per Exposure")
plt.title("Actual vs. Predicted Annual Cost by Risk Decile")
plt.xticks(x_indices, decile_summary["decile"])
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: Cumulative Loss Capture Curve
df_sorted = df_eval.sort_values("pred_annual_cost", ascending=False).copy()
df_sorted["cum_exposure"] = df_sorted["exposure_time"].cumsum()
df_sorted["cum_actual_cost"] = df_sorted["annual_claim_cost"].cumsum()

cum_exp_pct = df_sorted["cum_exposure"].values / df_sorted["exposure_time"].sum()
cum_cost_pct = df_sorted["cum_actual_cost"].values / df_sorted["annual_claim_cost"].sum()

df_perfect = df_eval.sort_values("annual_claim_cost", ascending=False).copy()
df_perfect["cum_exposure"] = df_perfect["exposure_time"].cumsum()
df_perfect["cum_actual_cost"].values
df_perfect["cum_actual_cost"] = df_perfect["annual_claim_cost"].cumsum()
perfect_exp_pct = df_perfect["cum_exposure"].values / df_perfect["exposure_time"].sum()
perfect_cost_pct = df_perfect["cum_actual_cost"].values / df_perfect["annual_claim_cost"].sum()

plt.figure(figsize=(8, 6))
plt.plot(cum_exp_pct, cum_cost_pct, label="Model Cumulative Loss Capture", color="#2980b9", linewidth=2.5)
plt.plot(perfect_exp_pct, perfect_cost_pct, label="Perfect Model", color="#27ae60", linestyle=":", linewidth=2)
plt.plot([0, 1], [0, 1], label="Random Model (Baseline)", color="#7f8c8d", linestyle="--", linewidth=1.5)
plt.xlabel("Cumulative Proportion of Exposure")
plt.ylabel("Cumulative Proportion of Actual Claim Cost")
plt.title("Cumulative Loss Capture Curve")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()